<a href="https://colab.research.google.com/github/sinhajiya/DSE318-NLP-Assignment-Solutions/blob/main/Assignment3/22161_jiyasinha_nlpassignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification Using RNN, LSTM, Transformer and Pretrained Language Model

## Initialization

In [ ]:
import os
import numpy as np
import pandas as pd
from random import randint, sample, seed, choice
import re
from collections import Counter
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from nltk.tokenize import word_tokenize
import torch.nn as nn
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt
from torch.nn.functional import log_softmax, pad
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback


In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import warnings
warnings.filterwarnings("ignore")
RUN_EXAMPLES = True

## GIT CLONE

In [ ]:
!git clone https://github.com/islnlp/Assignment_1_2025

Cloning into 'Assignment_1_2025'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 35 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 1.06 MiB | 3.28 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [ ]:
!git clone https://github.com/sinhajiya/DSE318-NLP-Assignment-Solutions

Cloning into 'DSE318-NLP-Assignment-Solutions'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 313 (delta 5), reused 11 (delta 0), pack-reused 259 (from 1)
Receiving objects: 100% (313/313), 351.01 MiB | 35.96 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [ ]:
!git clone https://github.com/TrigonaMinima/HinglishNLP

Cloning into 'HinglishNLP'...
remote: Enumerating objects: 156, done.
remote: Total 156 (delta 0), reused 0 (delta 0), pack-reused 156 (from 1)
Receiving objects: 100% (156/156), 947.84 KiB | 2.38 MiB/s, done.
Resolving deltas: 100% (53/53), done.


## Loading data and preprocessing

In [ ]:
def load_data(name):
  root_fp = f"/content/Assignment_1_2025/{name}"
  train = pd.read_csv(os.path.join(root_fp, "train.csv"))
  test = pd.read_csv(os.path.join(root_fp, "val.csv"))
  train = train.dropna(subset=['Sentence'])
  test = test.dropna(subset=['Sentence'])
  return train, test

In [ ]:
def preprocess_text(Sentence):

    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*(),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    Sentence = Sentence.lower()
    Sentence = re.sub(url_pattern, "", Sentence)
    Sentence = re.sub(r"\.{2,}", ".", Sentence)
    Sentence = re.sub(r"\s+", " ", Sentence).strip()
    Sentence = re.sub(r"[^a-zA-Z\s]", "", Sentence)
    return Sentence

In [ ]:
def remove_stop_words(data):
    hinglish_stop_words = pd.read_csv('/content/HinglishNLP/data/assets/stop_hinglish', sep=" ", header=None)
    stop_words = set(hinglish_stop_words[0].to_list())
    filtered_sentences = []

    for sentence in data['Sentence_preprocessed']:
        word_tokens = word_tokenize(sentence)
        filtered_words = [word for word in word_tokens if word not in stop_words]
        filtered_sentences.append(" ".join(filtered_words))

    data['Sentence_preprocessed'] = filtered_sentences
    return data


In [ ]:
def load_and_preprocess_data(name):
  print("Loading the data..\n")
  train, test = load_data(name)
  print("Performing the preprocessing steps:\n 1. All lower case characters \n2. URL removal\n3. Multiple dots to single dot\n4. Extra spaces to single space\n5. Removes non-alphabetic chars.\n")
  train["Sentence_preprocessed"] = train["Sentence"].astype(str).apply(preprocess_text)
  test["Sentence_preprocessed"] = test["Sentence"].astype(str).apply(preprocess_text)
  print("Removing the stop words..\n")
  train = remove_stop_words(train)
  test = remove_stop_words(test)
  print(f"Loaded and preprocessed {name} dataset.\n")
  return train, test

## Creating vocabulary

In [ ]:

def form_vocab(data,isdataframe=True):

  word2index = dict()  # Gives mapping from index to word
  index2word = dict()  # Gives mapping of word to index
  word2index = {'OOV': 0}
  index2word = {0: 'OOV'}
  vocab_size = 1
  vocab = {'OOV'}

  if isdataframe:
    data = data["Sentence_preprocessed"]

  for sentence in data:
    for word in sentence.split():
      if word not in word2index:
        word2index[word] = vocab_size
        index2word[vocab_size] = word
        vocab.add(word)
        vocab_size += 1
  print(f"Vocabulary of {vocab_size} created")
  return vocab, vocab_size, word2index, index2word

## Prepare Dataset for Torch

In [ ]:
class data_torch(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)

In [ ]:
def pad(batch):
  sequences, labels = zip(*batch)
  padded_seq = pad_sequence(sequences, batch_first=True, padding_value = 0)
  labels = torch.stack(labels)
  return padded_seq, labels

In [ ]:
def prepare_data(data, word2index):

  X = data['Sentence_preprocessed']
  y = data['Tag']
  X = [[word2index.get(word, word2index['OOV']) for word in sentence.split()] for sentence in X]

  data = data_torch(X, y)
  print("Prepared data for the model.\n")
  return data


In [ ]:
def get_all_items(name):
    train, test = load_and_preprocess_data(name)
    train_data, val_data = train_test_split(train, test_size=0.3,stratify=train['Tag'], random_state=42)
    train_data = train_data.reset_index(drop=True)
    val_data = val_data.reset_index(drop=True)
    vocab, vocab_size, word2index, index2word = form_vocab(train, isdataframe=True)
    print("Preparing the training data..\n")
    train_dataset = prepare_data(train_data, word2index)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=pad)
    print("Preparing the val data..\n")
    val_dataset = prepare_data(val_data, word2index)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, collate_fn=pad)
    print("Preparing the test data..\n")
    test_dataset = prepare_data(test, word2index)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, collate_fn=pad)

    print(f"\nReturning all data and settings for the '{name}' dataset in the following manner:\n train, val,test, vocab, vocab_size, word2index, index2word, train_dataset,train_loader,val_dataset,val_loader, test_dataset and test_loader")

    return {
        f"train": train_data,
        f"val": val_data,
        f"test": test,
        f"vocab": vocab,
        f"vocab_size": vocab_size,
        f"word2index": word2index,
        f"index2word": index2word,
        f"train_dataset": train_dataset,
        f"train_loader": train_loader,
        f"val_dataset": val_dataset,
        f"val_loader": val_loader,
        f"test_dataset": test_dataset,
        f"test_loader": test_loader
    }

## Training functions

trains the model and store the model along with accuracy plot

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        if self.best_model_state:
            model.load_state_dict(self.best_model_state)


In [ ]:
def train_classifier(name, model_class, train_loader, val_loader, batch_size=32, num_epochs=30, **kwargs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    name_model = model_class.__name__.lower()
    print(f"Training the {name_model} model...\n")

    model = model_class(**kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    all_labels = []
    for _, y_batch in train_loader:
        all_labels.extend(y_batch.numpy())
    all_labels = np.array(all_labels)

    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(all_labels), y=all_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    loss_values = []
    accuracy_values = []

    early_stopping = EarlyStopping(patience=10, delta=0.001)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct_preds = 0
        total_samples = 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct_preds += (predicted == y_batch).sum().item()
            total_samples += y_batch.size(0)

        avg_loss = total_loss / len(train_loader)
        accuracy = correct_preds / total_samples
        loss_values.append(avg_loss)
        accuracy_values.append(accuracy)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}")

        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}.\n")
            break

    early_stopping.load_best_model(model)

    dir = f"/content/{name}"
    os.makedirs(dir, exist_ok=True)
    model_path = os.path.join(dir, f"{name}_{name_model}_model.pt")
    torch.save(model.state_dict(), model_path)

    epochs = range(1, len(loss_values) + 1)
    fig, ax1 = plt.subplots()
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color="tab:red")
    ax1.plot(epochs, loss_values, color="tab:red", label="Loss")
    ax1.tick_params(axis='y', labelcolor="tab:red")
    ax2 = ax1.twinx()
    ax2.set_ylabel("Accuracy", color="tab:blue")
    ax2.plot(epochs, accuracy_values, color="tab:blue", label="Accuracy")
    ax2.tick_params(axis='y', labelcolor="tab:blue")
    fig.tight_layout()
    plt.title(f"Loss & Accuracy Curve for {name}")
    plt.grid(True)
    plt.savefig(os.path.join(dir, f"{name}_{name_model}_loss_accuracy_plot.png"))
    plt.close()

    print(f"Training completed.\nSaved the model and its loss+accuracy plot at {dir}.\n")


## RNN Classifier

In [ ]:
class RNN(nn.Module):
  def __init__(self, vocab_size,embedding_len = 100,hidden_dim=128,num_layers=2):
    super(RNN, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_len)
    self.rnn = nn.RNN(input_size=embedding_len, hidden_size = hidden_dim, num_layers=num_layers, bidirectional=True, batch_first = True)
    self.dropout = nn.Dropout(0.5)
    self.fc = nn.Linear((hidden_dim * 2), 2)

  def forward(self, X_batch):
    embeddings = self.embedding(X_batch)
    output, hidden_layer = self.rnn(embeddings)
    f = hidden_layer[-2,:,:]
    b = hidden_layer[-1,:,:]
    last_hidden_layer = torch.cat((f,b), dim=1)
    return self.fc(self.dropout(last_hidden_layer))


## LSTM classifier

In [ ]:
class LSTM(nn.Module):
  def __init__(self, vocab_size,embedding_len = 100,hidden_dim=128,num_layers=2):
    super(LSTM, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_len)
    self.lstm = nn.LSTM(input_size=embedding_len, hidden_size = hidden_dim, num_layers=num_layers, bidirectional=True, batch_first = True)
    self.dropout = nn.Dropout(0.5)
    self.fc = nn.Linear((hidden_dim * 2), 2)

  def forward(self, X_batch):
    embeddings = self.embedding(X_batch)
    output, (h_n, c_n) = self.lstm(embeddings)
    f = h_n[-2, :, :]
    b = h_n[-1, :, :]
    last_hidden_layer = torch.cat((f, b), dim=1)

    return self.fc(self.dropout(last_hidden_layer))



## Transformer Classifier

In [ ]:
class TRANSFORMER(nn.Module):
  def __init__(self,vocab_size, embedding_len=100,num_heads=4, num_layers=2,num_classes=2, max_len=512, dropout=0.1):
    super(TRANSFORMER, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_len)
    self.positional_encoder = nn.Parameter(torch.zeros(1,max_len, embedding_len))
    encoder_layer = nn.TransformerEncoderLayer(
        d_model = embedding_len,
        nhead = num_heads,
        dim_feedforward = embedding_len*4,
        dropout = dropout,
        batch_first = True
    )

    self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
    self.dropout = nn.Dropout(dropout)
    self.fc = nn.Linear(embedding_len, num_classes)

  def forward(self, X_batch):
    seq_len = X_batch.size(1)
    x=self.embedding(X_batch)
    x+=self.positional_encoder[:,:seq_len,:]
    x=self.transformer_encoder(x)
    x=x.mean(dim=1)
    x=self.dropout(x)
    out=self.fc(x)
    return out


## Evaluation

In [ ]:
def load_model(name, model_class,**kwargs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    name_model = model_class.__name__.lower()
    print(f"Loading the {name_model} model...\n")
    model = model_class(**kwargs)
    model_path = f"/content/DSE318-NLP-Assignment-Solutions/Assignment3/{name}/{name}_{name_model}_model.pt"
    # model_path = f"/content/{name}/{name}_{name_model}_model.pt"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

In [ ]:
def evaluate(name, model_class, word2index, test_loader, batch_size=32, threshold=0.5, **kwargs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_names = [name, f"non_{name}"]

    model = load_model(name, model_class, **kwargs)
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = (probs > threshold).long()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=target_names))


## Get all items

In [ ]:
hate = get_all_items('hate')

Loading the data..

Performing the preprocessing steps:
 1. All lower case characters 
2. URL removal
3. Multiple dots to single dot
4. Extra spaces to single space
5. Removes non-alphabetic chars.

Removing the stop words..

Loaded and preprocessed hate dataset.

Vocabulary of 12283 created
Preparing the training data..

Prepared data for the model.

Preparing the val data..

Prepared data for the model.

Preparing the test data..

Prepared data for the model.


Returning all data and settings for the 'hate' dataset in the following manner:
 train, val,test, vocab, vocab_size, word2index, index2word, train_dataset,train_loader,val_dataset,val_loader, test_dataset and test_loader


In [ ]:
sarcasm = get_all_items('sarcasm')


Loading the data..

Performing the preprocessing steps:
 1. All lower case characters 
2. URL removal
3. Multiple dots to single dot
4. Extra spaces to single space
5. Removes non-alphabetic chars.

Removing the stop words..

Loaded and preprocessed sarcasm dataset.

Vocabulary of 13908 created
Preparing the training data..

Prepared data for the model.

Preparing the val data..

Prepared data for the model.

Preparing the test data..

Prepared data for the model.


Returning all data and settings for the 'sarcasm' dataset in the following manner:
 train, val,test, vocab, vocab_size, word2index, index2word, train_dataset,train_loader,val_dataset,val_loader, test_dataset and test_loader


In [ ]:
humor = get_all_items('humor')

Loading the data..

Performing the preprocessing steps:
 1. All lower case characters 
2. URL removal
3. Multiple dots to single dot
4. Extra spaces to single space
5. Removes non-alphabetic chars.

Removing the stop words..

Loaded and preprocessed humor dataset.

Vocabulary of 6607 created
Preparing the training data..

Prepared data for the model.

Preparing the val data..

Prepared data for the model.

Preparing the test data..

Prepared data for the model.


Returning all data and settings for the 'humor' dataset in the following manner:
 train, val,test, vocab, vocab_size, word2index, index2word, train_dataset,train_loader,val_dataset,val_loader, test_dataset and test_loader


## RNNs

### Hate dataset

In [ ]:
# train_classifier('hate', RNN, hate['train_loader'], hate['val_loader'],batch_size=32, num_epochs = 30,vocab_size=hate['vocab_size'])


Training the rnn model...

Epoch 1/30, Train Loss: 0.7120, Accuracy: 0.5496
Epoch 1, Val Loss: 0.7057
Epoch 2/30, Train Loss: 0.6612, Accuracy: 0.6116
Epoch 2, Val Loss: 0.6880
Epoch 3/30, Train Loss: 0.5735, Accuracy: 0.7135
Epoch 3, Val Loss: 0.7395
Epoch 4/30, Train Loss: 0.4779, Accuracy: 0.7767
Epoch 4, Val Loss: 0.8548
Epoch 5/30, Train Loss: 0.3855, Accuracy: 0.8349
Epoch 5, Val Loss: 0.9784
Epoch 6/30, Train Loss: 0.2574, Accuracy: 0.8989
Epoch 6, Val Loss: 1.1768
Epoch 7/30, Train Loss: 0.1638, Accuracy: 0.9387
Epoch 7, Val Loss: 1.4576
Epoch 8/30, Train Loss: 0.1038, Accuracy: 0.9672
Epoch 8, Val Loss: 1.8180
Epoch 9/30, Train Loss: 0.1100, Accuracy: 0.9567
Epoch 9, Val Loss: 2.0146
Epoch 10/30, Train Loss: 0.0897, Accuracy: 0.9649
Epoch 10, Val Loss: 1.9436
Epoch 11/30, Train Loss: 0.0476, Accuracy: 0.9832
Epoch 11, Val Loss: 2.2084
Epoch 12/30, Train Loss: 0.0230, Accuracy: 0.9926
Epoch 12, Val Loss: 2.3660
Early stopping at epoch 12.

Training completed.
Saved the model an

In [ ]:
evaluate('hate', RNN, hate['word2index'], hate['test_loader'], batch_size=32,vocab_size=hate['vocab_size'])

Loading the rnn model...


Classification Report:

              precision    recall  f1-score   support

        hate       0.69      0.74      0.71       309
    non_hate       0.36      0.30      0.33       148

    accuracy                           0.60       457
   macro avg       0.52      0.52      0.52       457
weighted avg       0.58      0.60      0.59       457



### Sarcasm dataset

In [ ]:
# train_classifier('sarcasm', RNN, sarcasm['train_loader'], sarcasm['val_loader'],batch_size=32, num_epochs = 30,vocab_size=sarcasm['vocab_size'])


Training the rnn model...

Epoch 1/30, Train Loss: 0.5965, Accuracy: 0.7078
Epoch 1, Val Loss: 0.3844
Epoch 2/30, Train Loss: 0.2893, Accuracy: 0.9024
Epoch 2, Val Loss: 0.3297
Epoch 3/30, Train Loss: 0.1995, Accuracy: 0.9313
Epoch 3, Val Loss: 0.4496
Epoch 4/30, Train Loss: 0.1654, Accuracy: 0.9357
Epoch 4, Val Loss: 0.3827
Epoch 5/30, Train Loss: 0.1180, Accuracy: 0.9619
Epoch 5, Val Loss: 0.4992
Epoch 6/30, Train Loss: 0.0492, Accuracy: 0.9810
Epoch 6, Val Loss: 0.6526
Epoch 7/30, Train Loss: 0.0706, Accuracy: 0.9697
Epoch 7, Val Loss: 0.8058
Epoch 8/30, Train Loss: 0.0317, Accuracy: 0.9861
Epoch 8, Val Loss: 0.8097
Epoch 9/30, Train Loss: 0.0195, Accuracy: 0.9935
Epoch 9, Val Loss: 0.8288
Epoch 10/30, Train Loss: 0.0107, Accuracy: 0.9952
Epoch 10, Val Loss: 0.9306
Epoch 11/30, Train Loss: 0.0036, Accuracy: 0.9990
Epoch 11, Val Loss: 1.0239
Epoch 12/30, Train Loss: 0.0024, Accuracy: 0.9997
Epoch 12, Val Loss: 1.0816
Early stopping at epoch 12.

Training completed.
Saved the model an

In [ ]:
evaluate('sarcasm', RNN, sarcasm['word2index'], sarcasm['test_loader'], batch_size=32, vocab_size=sarcasm['vocab_size'])


Loading the rnn model...


Classification Report:

              precision    recall  f1-score   support

     sarcasm       0.93      0.98      0.95       474
 non_sarcasm       0.61      0.27      0.38        51

    accuracy                           0.91       525
   macro avg       0.77      0.63      0.67       525
weighted avg       0.90      0.91      0.90       525



### Humor

In [ ]:
# train_classifier('humor', RNN, humor['train_loader'], humor['val_loader'],batch_size=32, num_epochs = 30,vocab_size=humor['vocab_size'])


Training the rnn model...

Epoch 1/30, Train Loss: 0.6893, Accuracy: 0.5914
Epoch 1, Val Loss: 0.6326
Epoch 2/30, Train Loss: 0.5894, Accuracy: 0.7016
Epoch 2, Val Loss: 0.6957
Epoch 3/30, Train Loss: 0.4997, Accuracy: 0.7627
Epoch 3, Val Loss: 0.7968
Epoch 4/30, Train Loss: 0.3955, Accuracy: 0.8245
Epoch 4, Val Loss: 0.9026
Epoch 5/30, Train Loss: 0.2828, Accuracy: 0.8783
Epoch 5, Val Loss: 1.1321
Epoch 6/30, Train Loss: 0.1702, Accuracy: 0.9340
Epoch 6, Val Loss: 1.4249
Epoch 7/30, Train Loss: 0.1160, Accuracy: 0.9576
Epoch 7, Val Loss: 1.5594
Epoch 8/30, Train Loss: 0.0833, Accuracy: 0.9697
Epoch 8, Val Loss: 1.8778
Epoch 9/30, Train Loss: 0.0467, Accuracy: 0.9861
Epoch 9, Val Loss: 2.1308
Epoch 10/30, Train Loss: 0.0269, Accuracy: 0.9939
Epoch 10, Val Loss: 2.2965
Epoch 11/30, Train Loss: 0.0158, Accuracy: 0.9958
Epoch 11, Val Loss: 2.3510
Early stopping at epoch 11.

Training completed.
Saved the model and its loss+accuracy plot at /content/humor.



In [ ]:
evaluate('humor', RNN, humor['word2index'], humor['test_loader'], batch_size=32, vocab_size=humor['vocab_size'])


Loading the rnn model...


Classification Report:

              precision    recall  f1-score   support

       humor       0.42      0.51      0.46       119
   non_humor       0.61      0.52      0.56       176

    accuracy                           0.52       295
   macro avg       0.52      0.52      0.51       295
weighted avg       0.54      0.52      0.52       295



## LSTMs

### Hate dataset

In [ ]:
# train_classifier('hate', LSTM, hate['train_loader'], hate['val_loader'],batch_size=32, num_epochs = 30,vocab_size=hate['vocab_size'])


Training the lstm model...

Epoch 1/30, Train Loss: 0.6813, Accuracy: 0.5734
Epoch 1, Val Loss: 0.6650
Epoch 2/30, Train Loss: 0.6294, Accuracy: 0.6472
Epoch 2, Val Loss: 0.6832
Epoch 3/30, Train Loss: 0.5433, Accuracy: 0.7315
Epoch 3, Val Loss: 0.7237
Epoch 4/30, Train Loss: 0.3928, Accuracy: 0.8232
Epoch 4, Val Loss: 0.8976
Epoch 5/30, Train Loss: 0.2155, Accuracy: 0.9173
Epoch 5, Val Loss: 1.2870
Epoch 6/30, Train Loss: 0.1122, Accuracy: 0.9598
Epoch 6, Val Loss: 1.6368
Epoch 7/30, Train Loss: 0.0667, Accuracy: 0.9754
Epoch 7, Val Loss: 1.9897
Epoch 8/30, Train Loss: 0.0375, Accuracy: 0.9848
Epoch 8, Val Loss: 2.0889
Epoch 9/30, Train Loss: 0.0417, Accuracy: 0.9844
Epoch 9, Val Loss: 2.2573
Epoch 10/30, Train Loss: 0.0182, Accuracy: 0.9949
Epoch 10, Val Loss: 2.6085
Epoch 11/30, Train Loss: 0.0105, Accuracy: 0.9969
Epoch 11, Val Loss: 2.6655
Early stopping at epoch 11.

Training completed.
Saved the model and its loss+accuracy plot at /content/hate.



In [ ]:
evaluate('hate', LSTM, hate['word2index'], hate['test_loader'], batch_size=32,vocab_size=hate['vocab_size'])


Loading the lstm model...


Classification Report:

              precision    recall  f1-score   support

        hate       0.69      0.78      0.73       309
    non_hate       0.36      0.26      0.30       148

    accuracy                           0.61       457
   macro avg       0.52      0.52      0.51       457
weighted avg       0.58      0.61      0.59       457



### Sarcasm dataset

In [ ]:
# train_classifier('sarcasm', LSTM, sarcasm['train_loader'], sarcasm['val_loader'],batch_size=32, num_epochs = 30,vocab_size=sarcasm['vocab_size'])


Training the lstm model...

Epoch 1/30, Train Loss: 0.4643, Accuracy: 0.8456
Epoch 1, Val Loss: 0.2568
Epoch 2/30, Train Loss: 0.1467, Accuracy: 0.9357
Epoch 2, Val Loss: 0.2722
Epoch 3/30, Train Loss: 0.0958, Accuracy: 0.9554
Epoch 3, Val Loss: 0.2538
Epoch 4/30, Train Loss: 0.0809, Accuracy: 0.9629
Epoch 4, Val Loss: 0.4481
Epoch 5/30, Train Loss: 0.0567, Accuracy: 0.9721
Epoch 5, Val Loss: 0.5726
Epoch 6/30, Train Loss: 0.0706, Accuracy: 0.9724
Epoch 6, Val Loss: 0.2467
Epoch 7/30, Train Loss: 0.0537, Accuracy: 0.9745
Epoch 7, Val Loss: 0.4088
Epoch 8/30, Train Loss: 0.0172, Accuracy: 0.9939
Epoch 8, Val Loss: 0.5047
Epoch 9/30, Train Loss: 0.0081, Accuracy: 0.9980
Epoch 9, Val Loss: 0.6100
Epoch 10/30, Train Loss: 0.0030, Accuracy: 0.9993
Epoch 10, Val Loss: 0.7563
Epoch 11/30, Train Loss: 0.0010, Accuracy: 1.0000
Epoch 11, Val Loss: 0.7292
Epoch 12/30, Train Loss: 0.0006, Accuracy: 1.0000
Epoch 12, Val Loss: 0.7788
Epoch 13/30, Train Loss: 0.0004, Accuracy: 1.0000
Epoch 13, Val Lo

In [ ]:
evaluate('sarcasm', LSTM, sarcasm['word2index'], sarcasm['test_loader'], batch_size=32,vocab_size=sarcasm['vocab_size'])


Loading the lstm model...


Classification Report:

              precision    recall  f1-score   support

     sarcasm       0.92      0.98      0.95       474
 non_sarcasm       0.59      0.25      0.36        51

    accuracy                           0.91       525
   macro avg       0.76      0.62      0.65       525
weighted avg       0.89      0.91      0.89       525



### Humor dataset

In [ ]:
# train_classifier('humor', LSTM, humor['train_loader'], humor['val_loader'],batch_size=32, num_epochs = 30,vocab_size=humor['vocab_size'])


Training the lstm model...

Epoch 1/30, Train Loss: 0.6711, Accuracy: 0.5847
Epoch 1, Val Loss: 0.6309
Epoch 2/30, Train Loss: 0.6087, Accuracy: 0.6628
Epoch 2, Val Loss: 0.6376
Epoch 3/30, Train Loss: 0.5201, Accuracy: 0.7415
Epoch 3, Val Loss: 0.7207
Epoch 4/30, Train Loss: 0.3550, Accuracy: 0.8462
Epoch 4, Val Loss: 0.9467
Epoch 5/30, Train Loss: 0.2049, Accuracy: 0.9201
Epoch 5, Val Loss: 1.1892
Epoch 6/30, Train Loss: 0.1134, Accuracy: 0.9528
Epoch 6, Val Loss: 1.8492
Epoch 7/30, Train Loss: 0.1100, Accuracy: 0.9576
Epoch 7, Val Loss: 1.6765
Epoch 8/30, Train Loss: 0.0422, Accuracy: 0.9837
Epoch 8, Val Loss: 2.1477
Epoch 9/30, Train Loss: 0.0370, Accuracy: 0.9879
Epoch 9, Val Loss: 2.1430
Epoch 10/30, Train Loss: 0.0178, Accuracy: 0.9958
Epoch 10, Val Loss: 2.4572
Epoch 11/30, Train Loss: 0.0131, Accuracy: 0.9952
Epoch 11, Val Loss: 2.6800
Early stopping at epoch 11.

Training completed.
Saved the model and its loss+accuracy plot at /content/humor.



In [ ]:
evaluate('humor', LSTM, humor['word2index'], humor['test_loader'], batch_size=32,vocab_size=humor['vocab_size'])


Loading the lstm model...


Classification Report:

              precision    recall  f1-score   support

       humor       0.46      0.83      0.59       119
   non_humor       0.75      0.35      0.47       176

    accuracy                           0.54       295
   macro avg       0.61      0.59      0.53       295
weighted avg       0.64      0.54      0.52       295



## Transformers

### Hate dataset

In [ ]:
# train_classifier('hate', TRANSFORMER, hate['train_loader'], hate['val_loader'],batch_size=32, num_epochs = 30,vocab_size=hate['vocab_size'])
evaluate('hate', TRANSFORMER, hate['word2index'], hate['test_loader'], batch_size=32,vocab_size=hate['vocab_size'])


Training the transformer model...

Epoch 1/30, Train Loss: 0.7179, Accuracy: 0.5726
Epoch 1, Val Loss: 0.6680
Epoch 2/30, Train Loss: 0.6598, Accuracy: 0.6237
Epoch 2, Val Loss: 0.6638
Epoch 3/30, Train Loss: 0.6086, Accuracy: 0.6803
Epoch 3, Val Loss: 0.6915
Epoch 4/30, Train Loss: 0.4916, Accuracy: 0.7564
Epoch 4, Val Loss: 0.7745
Epoch 5/30, Train Loss: 0.3596, Accuracy: 0.8419
Epoch 5, Val Loss: 0.9046
Epoch 6/30, Train Loss: 0.2381, Accuracy: 0.9009
Epoch 6, Val Loss: 1.1580
Epoch 7/30, Train Loss: 0.1581, Accuracy: 0.9379
Epoch 7, Val Loss: 1.5501
Epoch 8/30, Train Loss: 0.1093, Accuracy: 0.9602
Epoch 8, Val Loss: 1.5452
Epoch 9/30, Train Loss: 0.0881, Accuracy: 0.9641
Epoch 9, Val Loss: 1.9012
Epoch 10/30, Train Loss: 0.0509, Accuracy: 0.9785
Epoch 10, Val Loss: 2.1708
Epoch 11/30, Train Loss: 0.0374, Accuracy: 0.9859
Epoch 11, Val Loss: 2.2299
Epoch 12/30, Train Loss: 0.0388, Accuracy: 0.9867
Epoch 12, Val Loss: 2.3838
Early stopping at epoch 12.

Training completed.
Saved the 

### Humor dataset

In [ ]:
# train_classifier('humor', TRANSFORMER, humor['train_loader'], humor['val_loader'],batch_size=32, num_epochs = 30,vocab_size=humor['vocab_size'])
evaluate('humor', TRANSFORMER, humor['word2index'], humor['test_loader'], batch_size=32,vocab_size=humor['vocab_size'])


Training the transformer model...

Epoch 1/30, Train Loss: 0.6718, Accuracy: 0.6096
Epoch 1, Val Loss: 0.6293
Epoch 2/30, Train Loss: 0.6238, Accuracy: 0.6477
Epoch 2, Val Loss: 0.6262
Epoch 3/30, Train Loss: 0.5780, Accuracy: 0.7040
Epoch 3, Val Loss: 0.6981
Epoch 4/30, Train Loss: 0.4605, Accuracy: 0.7893
Epoch 4, Val Loss: 0.7936
Epoch 5/30, Train Loss: 0.3263, Accuracy: 0.8596
Epoch 5, Val Loss: 1.0312
Epoch 6/30, Train Loss: 0.2015, Accuracy: 0.9183
Epoch 6, Val Loss: 1.4187
Epoch 7/30, Train Loss: 0.1368, Accuracy: 0.9479
Epoch 7, Val Loss: 1.5388
Epoch 8/30, Train Loss: 0.0984, Accuracy: 0.9625
Epoch 8, Val Loss: 2.0400
Epoch 9/30, Train Loss: 0.0563, Accuracy: 0.9818
Epoch 9, Val Loss: 2.0464
Epoch 10/30, Train Loss: 0.0325, Accuracy: 0.9909
Epoch 10, Val Loss: 2.3643
Epoch 11/30, Train Loss: 0.0243, Accuracy: 0.9915
Epoch 11, Val Loss: 2.6410
Epoch 12/30, Train Loss: 0.0254, Accuracy: 0.9903
Epoch 12, Val Loss: 2.5128
Early stopping at epoch 12.

Training completed.
Saved the 

### Sarcasm dataset

In [ ]:
# train_classifier('sarcasm', TRANSFORMER, sarcasm['train_loader'], sarcasm['val_loader'],batch_size=32, num_epochs = 30,vocab_size=sarcasm['vocab_size'])
evaluate('sarcasm', TRANSFORMER, sarcasm['word2index'], sarcasm['test_loader'], batch_size=32,vocab_size=sarcasm['vocab_size'])


Training the transformer model...

Epoch 1/30, Train Loss: 0.4112, Accuracy: 0.8286
Epoch 1, Val Loss: 0.1769
Epoch 2/30, Train Loss: 0.1512, Accuracy: 0.9282
Epoch 2, Val Loss: 0.1411
Epoch 3/30, Train Loss: 0.1119, Accuracy: 0.9497
Epoch 3, Val Loss: 0.2295
Epoch 4/30, Train Loss: 0.1121, Accuracy: 0.9483
Epoch 4, Val Loss: 0.1944
Epoch 5/30, Train Loss: 0.1052, Accuracy: 0.9568
Epoch 5, Val Loss: 0.2819
Epoch 6/30, Train Loss: 0.0763, Accuracy: 0.9680
Epoch 6, Val Loss: 0.3766
Epoch 7/30, Train Loss: 0.0669, Accuracy: 0.9701
Epoch 7, Val Loss: 0.3099
Epoch 8/30, Train Loss: 0.0525, Accuracy: 0.9782
Epoch 8, Val Loss: 0.3483
Epoch 9/30, Train Loss: 0.0305, Accuracy: 0.9874
Epoch 9, Val Loss: 0.5528
Epoch 10/30, Train Loss: 0.0286, Accuracy: 0.9888
Epoch 10, Val Loss: 0.3511
Epoch 11/30, Train Loss: 0.0133, Accuracy: 0.9949
Epoch 11, Val Loss: 0.5488
Epoch 12/30, Train Loss: 0.0066, Accuracy: 0.9976
Epoch 12, Val Loss: 0.6394
Early stopping at epoch 12.

Training completed.
Saved the 

## BERT Classifier

In [ ]:
model_name = "l3cube-pune/hing-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/716 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hing-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def BERT_tokenize(Sentence, Tag):
    encodings = tokenizer(list(Sentence), truncation=True, padding=True)
    return encodings, list(Tag)

In [ ]:
class HinglishTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
def prepare_data_for_BERT(item_dict):
    train = item_dict['train']
    val = item_dict['val']
    test = item_dict['test']

    train_enc, train_labels = BERT_tokenize(train['Sentence'], train['Tag'])
    val_enc, val_labels = BERT_tokenize(val['Sentence'], val['Tag'])
    test_enc, test_labels = BERT_tokenize(test['Sentence'], test['Tag'])

    train_dataset = HinglishTextDataset(train_enc, train_labels)
    val_dataset = HinglishTextDataset(val_enc, val_labels)
    test_dataset = HinglishTextDataset(test_enc, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    return {
    f"BERT_train_loader": train_loader,
    f"BERT_val_loader": val_loader,
    f"BERT_test_loader": test_loader
}


In [ ]:
def train_bert_classifier(name, model, train_loader, val_loader, num_epochs=30):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Fine tuning the BERT model...\n")
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

    all_labels = []
    for batch in train_loader:
        all_labels.extend(batch['labels'].to(device).cpu().numpy())
    all_labels = np.array(all_labels)

    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(all_labels), y=all_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    loss_values = []
    accuracy_values = []

    early_stopping = EarlyStopping(patience=10, delta=0.001)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct_preds = 0
        total_samples = 0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct_preds += (preds == batch['labels']).sum().item()
            total_samples += batch['labels'].size(0)

        avg_loss = total_loss / len(train_loader)
        accuracy = correct_preds / total_samples
        loss_values.append(avg_loss)
        accuracy_values.append(accuracy)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1}, Val Loss: {avg_val_loss:.4f}")

        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}.\n")
            break

    early_stopping.load_best_model(model)

    # Save model + plot
    dir = f"/content/{name}"
    os.makedirs(dir, exist_ok=True)
    model_path = os.path.join(dir, f"{name}_bert_model.pt")
    torch.save(model.state_dict(), model_path)

    epochs = range(1, len(loss_values) + 1)
    fig, ax1 = plt.subplots()
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color="tab:red")
    ax1.plot(epochs, loss_values, color="tab:red", label="Loss")
    ax1.tick_params(axis='y', labelcolor="tab:red")
    ax2 = ax1.twinx()
    ax2.set_ylabel("Accuracy", color="tab:blue")
    ax2.plot(epochs, accuracy_values, color="tab:blue", label="Accuracy")
    ax2.tick_params(axis='y', labelcolor="tab:blue")
    fig.tight_layout()
    plt.title(f"Loss & Accuracy Curve for {name} (BERT)")
    plt.grid(True)
    plt.savefig(os.path.join(dir, f"{name}_bert_loss_accuracy_plot.png"))
    plt.close()

    print(f"Training completed.\nSaved the BERT model and plot at {dir}.\n")


In [ ]:
def load_bert_model(name, model_name="l3cube-pune/hing-bert"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading the BERT model...\n")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model_path = f"/content/{name}/{name}_bert_model.pt"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

In [ ]:
def evaluate_bert(name, test_loader, batch_size=32, threshold=0.5, model_name="l3cube-pune/hing-bert"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_names = [name, f"non_{name}"]
    model = load_bert_model(name, model_name)

    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = (probs > threshold).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=target_names))


## BERT

### Hate dataset

In [ ]:
hate_BERT = prepare_data_for_BERT(hate)
# train_bert_classifier('hate', bert_model, train_loader=hate_BERT['BERT_train_loader'], val_loader=hate_BERT['BERT_val_loader'], num_epochs=30)


Fine tuning the BERT model...

Epoch 1/30, Train Loss: 0.6023, Accuracy: 0.6768
Epoch 1, Val Loss: 0.5719
Epoch 2/30, Train Loss: 0.5037, Accuracy: 0.7502
Epoch 2, Val Loss: 0.5866
Epoch 3/30, Train Loss: 0.3770, Accuracy: 0.8357
Epoch 3, Val Loss: 0.6712
Epoch 4/30, Train Loss: 0.2064, Accuracy: 0.9243
Epoch 4, Val Loss: 0.9003
Epoch 5/30, Train Loss: 0.1024, Accuracy: 0.9657
Epoch 5, Val Loss: 1.1077
Epoch 6/30, Train Loss: 0.0531, Accuracy: 0.9828
Epoch 6, Val Loss: 1.3925
Epoch 7/30, Train Loss: 0.0453, Accuracy: 0.9875
Epoch 7, Val Loss: 1.8124
Epoch 8/30, Train Loss: 0.0586, Accuracy: 0.9817
Epoch 8, Val Loss: 1.4253
Epoch 9/30, Train Loss: 0.0285, Accuracy: 0.9910
Epoch 9, Val Loss: 1.5633
Epoch 10/30, Train Loss: 0.0742, Accuracy: 0.9762
Epoch 10, Val Loss: 1.4374
Epoch 11/30, Train Loss: 0.0094, Accuracy: 0.9977
Epoch 11, Val Loss: 1.7151
Early stopping at epoch 11.

Training completed.
Saved the BERT model and plot at /content/hate.

Loading the BERT model...



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hing-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Classification Report:

              precision    recall  f1-score   support

        hate       0.79      0.70      0.74       309
    non_hate       0.49      0.61      0.54       148

    accuracy                           0.67       457
   macro avg       0.64      0.65      0.64       457
weighted avg       0.69      0.67      0.68       457



In [ ]:
evaluate_bert(name='hate', test_loader=hate_BERT['BERT_test_loader'], batch_size=32)


Loading the BERT model...



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hing-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Classification Report:

              precision    recall  f1-score   support

        hate       0.79      0.70      0.74       309
    non_hate       0.49      0.61      0.54       148

    accuracy                           0.67       457
   macro avg       0.64      0.65      0.64       457
weighted avg       0.69      0.67      0.68       457



### Sarcasm dataset

In [ ]:
sarcasm_BERT = prepare_data_for_BERT(sarcasm)
# train_bert_classifier('sarcasm', bert_model, train_loader=sarcasm_BERT['BERT_train_loader'], val_loader=sarcasm_BERT['BERT_val_loader'], num_epochs=30)


Fine tuning the BERT model...

Epoch 1/30, Train Loss: 0.2795, Accuracy: 0.9286
Epoch 1, Val Loss: 0.1497
Epoch 2/30, Train Loss: 0.0938, Accuracy: 0.9626
Epoch 2, Val Loss: 0.0903
Epoch 3/30, Train Loss: 0.0576, Accuracy: 0.9793
Epoch 3, Val Loss: 0.1378
Epoch 4/30, Train Loss: 0.0273, Accuracy: 0.9901
Epoch 4, Val Loss: 0.1509
Epoch 5/30, Train Loss: 0.0128, Accuracy: 0.9963
Epoch 5, Val Loss: 0.1791
Epoch 6/30, Train Loss: 0.0054, Accuracy: 0.9983
Epoch 6, Val Loss: 0.2132
Epoch 7/30, Train Loss: 0.0027, Accuracy: 0.9997
Epoch 7, Val Loss: 0.2050
Epoch 8/30, Train Loss: 0.0092, Accuracy: 0.9976
Epoch 8, Val Loss: 0.1880
Epoch 9/30, Train Loss: 0.0089, Accuracy: 0.9966
Epoch 9, Val Loss: 0.1728
Epoch 10/30, Train Loss: 0.0037, Accuracy: 0.9990
Epoch 10, Val Loss: 0.2166
Epoch 11/30, Train Loss: 0.0028, Accuracy: 0.9997
Epoch 11, Val Loss: 0.2331
Epoch 12/30, Train Loss: 0.0032, Accuracy: 0.9983
Epoch 12, Val Loss: 0.2290
Early stopping at epoch 12.

Training completed.
Saved the BERT

In [ ]:
evaluate_bert(name='sarcasm', test_loader=sarcasm_BERT['BERT_test_loader'], batch_size=32)


Loading the BERT model...



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hing-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Classification Report:

              precision    recall  f1-score   support

     sarcasm       0.99      0.97      0.98       474
 non_sarcasm       0.78      0.90      0.84        51

    accuracy                           0.97       525
   macro avg       0.88      0.94      0.91       525
weighted avg       0.97      0.97      0.97       525



### Humor dataset

In [ ]:
humor_BERT = prepare_data_for_BERT(humor)
# train_bert_classifier('humor', bert_model, train_loader=humor_BERT['BERT_train_loader'], val_loader=humor_BERT['BERT_val_loader'], num_epochs=30)


Fine tuning the BERT model...

Epoch 1/30, Train Loss: 0.7826, Accuracy: 0.6350
Epoch 1, Val Loss: 0.5882
Epoch 2/30, Train Loss: 0.5384, Accuracy: 0.7391
Epoch 2, Val Loss: 0.5963
Epoch 3/30, Train Loss: 0.4451, Accuracy: 0.7942
Epoch 3, Val Loss: 0.6777
Epoch 4/30, Train Loss: 0.3140, Accuracy: 0.8741
Epoch 4, Val Loss: 0.8796
Epoch 5/30, Train Loss: 0.1901, Accuracy: 0.9268
Epoch 5, Val Loss: 1.0651
Epoch 6/30, Train Loss: 0.0980, Accuracy: 0.9685
Epoch 6, Val Loss: 1.2080
Epoch 7/30, Train Loss: 0.0696, Accuracy: 0.9740
Epoch 7, Val Loss: 1.4366
Epoch 8/30, Train Loss: 0.0459, Accuracy: 0.9818
Epoch 8, Val Loss: 1.5989
Epoch 9/30, Train Loss: 0.0606, Accuracy: 0.9831
Epoch 9, Val Loss: 1.4340
Epoch 10/30, Train Loss: 0.0407, Accuracy: 0.9891
Epoch 10, Val Loss: 1.5276
Epoch 11/30, Train Loss: 0.0271, Accuracy: 0.9915
Epoch 11, Val Loss: 1.7456
Early stopping at epoch 11.

Training completed.
Saved the BERT model and plot at /content/humor.



In [ ]:
evaluate_bert(name='humor', test_loader=humor_BERT['BERT_test_loader'], batch_size=32)


Loading the BERT model...



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hing-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Classification Report:

              precision    recall  f1-score   support

       humor       0.57      0.61      0.59       119
   non_humor       0.72      0.69      0.71       176

    accuracy                           0.66       295
   macro avg       0.65      0.65      0.65       295
weighted avg       0.66      0.66      0.66       295



In [ ]:
!zip -r /content/hate.zip /content/hate

  adding: content/hate/ (stored 0%)
  adding: content/hate/hate_bert_model.pt (deflated 7%)
  adding: content/hate/hate_rnn_loss_accuracy_plot.png (deflated 5%)
  adding: content/hate/hate_transformer_loss_accuracy_plot.png (deflated 5%)
  adding: content/hate/hate_transformer_model.pt (deflated 10%)
  adding: content/hate/hate_lstm_model.pt (deflated 8%)
  adding: content/hate/hate_lstm_loss_accuracy_plot.png (deflated 6%)
  adding: content/hate/hate_rnn_model.pt (deflated 8%)
  adding: content/hate/hate_bert_loss_accuracy_plot.png (deflated 6%)


In [ ]:
!zip -r /content/homor.zip /content/humor

  adding: content/humor/ (stored 0%)
  adding: content/humor/humor_transformer_loss_accuracy_plot.png (deflated 5%)
  adding: content/humor/humor_lstm_loss_accuracy_plot.png (deflated 5%)
  adding: content/humor/humor_rnn_loss_accuracy_plot.png (deflated 6%)
  adding: content/humor/humor_bert_model.pt (deflated 7%)
  adding: content/humor/humor_transformer_model.pt (deflated 12%)
  adding: content/humor/humor_bert_loss_accuracy_plot.png (deflated 6%)
  adding: content/humor/humor_rnn_model.pt (deflated 8%)
  adding: content/humor/humor_lstm_model.pt (deflated 8%)


In [ ]:
!zip -r /content/sarcasm.zip /content/sarcasm

  adding: content/sarcasm/ (stored 0%)
  adding: content/sarcasm/sarcasm_bert_loss_accuracy_plot.png (deflated 7%)
  adding: content/sarcasm/sarcasm_lstm_loss_accuracy_plot.png (deflated 7%)
  adding: content/sarcasm/sarcasm_transformer_loss_accuracy_plot.png (deflated 6%)
  adding: content/sarcasm/sarcasm_bert_model.pt (deflated 7%)
  adding: content/sarcasm/sarcasm_transformer_model.pt (deflated 10%)
  adding: content/sarcasm/sarcasm_lstm_model.pt (deflated 8%)
  adding: content/sarcasm/sarcasm_rnn_model.pt (deflated 8%)
  adding: content/sarcasm/sarcasm_rnn_loss_accuracy_plot.png (deflated 5%)
